In [1]:
# NORTHSTAR — Solace Browser: Schedule Page QA
# DNA: schedule_qa = tabs(4) x a11y(ARIA+WCAG) x i18n(data-i18n) x security(CSP+escapeHtml) x structure(HTML5)
# Template: solace-cli/notebooks/qa-templates/template-page-qa.ipynb
# Towers: T1(Masters) T6(Hackers) T8(Elders) T9(World) T23(Web)
# Auth: 65537 | Papers: 46,47,49,50
#
# File-based probes — reads schedule.html + schedule-*.js + schedule.css directly
# REAL assertions. No mocks. No stubs.

import json, re, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "solace-browser-schedule-page-qa"
NOTEBOOK_PATH = "notebooks/qa/01-schedule-page-qa.ipynb"
PROJECT = "solace-browser"
MIN_SCORE = 70

BROWSER = Path("/home/phuc/projects/solace-browser")
WEB = BROWSER / "web"
LOCALES_DIR = BROWSER / "app" / "locales" / "yinyang"

# Schedule page sources
SCHEDULE_HTML = (WEB / "schedule.html").read_text(encoding="utf-8")
JS_DIR = WEB / "js"
CSS_DIR = WEB / "css"

SCHEDULE_JS_FILES = {
    "schedule-core": JS_DIR / "schedule-core.js",
    "schedule-approvals": JS_DIR / "schedule-approvals.js",
    "schedule-calendar": JS_DIR / "schedule-calendar.js",
    "schedule-evidence": JS_DIR / "schedule-evidence.js",
    "schedule-cloud": JS_DIR / "schedule-cloud.js",
}

results = []

def record(probe_id, name, passed, detail="", tower_ref=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status,
                    "detail": detail, "tower_ref": tower_ref})
    print(f"  {status}: {name}" + (f" -- {detail}" if detail and not passed else ""))

def read_js(name):
    return SCHEDULE_JS_FILES[name].read_text(encoding="utf-8")

print(f"BOOTSTRAP: schedule.html = {len(SCHEDULE_HTML)} chars")
print(f"JS modules: {list(SCHEDULE_JS_FILES.keys())}")
print(f"All JS exist: {all(f.exists() for f in SCHEDULE_JS_FILES.values())}")

BOOTSTRAP: schedule.html = 16783 chars
JS modules: ['schedule-core', 'schedule-approvals', 'schedule-calendar', 'schedule-evidence', 'schedule-cloud']
All JS exist: True


In [2]:
# PROBE 1: HTML Structure — doctype, charset, title, viewport, CSP, SEO meta
# Tower ref: T23 F3 (SEO), T6 F8 (Security Headers)
print("=" * 60)
print("PROBE 1: Schedule Page HTML Structure")
print("=" * 60)

src = SCHEDULE_HTML

# Basic HTML structure
record("P1-doctype", "has <!DOCTYPE html>",
       src.strip().lower().startswith("<!doctype html"), tower_ref="T23-F3")
record("P1-charset", "has charset",
       'charset' in src.lower(), tower_ref="T23-F3")
record("P1-title", "has <title>",
       bool(re.search(r'<title[^>]*>', src)), tower_ref="T23-F3")
record("P1-viewport", "has viewport meta",
       'viewport' in src, tower_ref="T23-F3")
record("P1-csp", "has Content-Security-Policy",
       'Content-Security-Policy' in src, tower_ref="T6-F8")

# SEO meta tags
record("P1-og-title", "has og:title",
       'og:title' in src, tower_ref="T23-F3")
record("P1-og-desc", "has og:description",
       'og:description' in src, tower_ref="T23-F3")
record("P1-canonical", "has canonical link",
       'rel="canonical"' in src, tower_ref="T23-F3")
record("P1-robots", "has robots meta (noindex for app page)",
       'robots' in src, tower_ref="T23-F3")

# Theme/layout loaded
record("P1-theme-js", "loads theme.js",
       'theme.js' in src, tower_ref="T23-F10")
record("P1-layout-js", "loads layout.js",
       'layout.js' in src, tower_ref="T23-F10")

# All 5 schedule JS modules loaded
for name in SCHEDULE_JS_FILES:
    record(f"P1-js-{name}", f"loads {name}.js",
           f"{name}.js" in src, tower_ref="T23-F10")

# CSS loaded
record("P1-site-css", "loads site.css",
       'site.css' in src, tower_ref="T23-F10")
record("P1-schedule-css", "loads schedule.css",
       'schedule.css' in src, tower_ref="T23-F10")

PROBE 1: Schedule Page HTML Structure
  PASS: has <!DOCTYPE html>
  PASS: has charset
  PASS: has <title>
  PASS: has viewport meta
  PASS: has Content-Security-Policy
  PASS: has og:title
  PASS: has og:description
  PASS: has canonical link
  PASS: has robots meta (noindex for app page)
  PASS: loads theme.js
  PASS: loads layout.js
  PASS: loads schedule-core.js
  PASS: loads schedule-approvals.js
  PASS: loads schedule-calendar.js
  PASS: loads schedule-evidence.js
  PASS: loads schedule-cloud.js
  PASS: loads site.css
  PASS: loads schedule.css


In [3]:
# PROBE 2: Accessibility — ARIA roles, tablist, tabpanel, dialog, sr-only, labels
# Tower ref: T8 (Elders), T1 F1 (Jony Ive — inclusive design)
print("=" * 60)
print("PROBE 2: Accessibility (WCAG)")
print("=" * 60)

src = SCHEDULE_HTML

# lang attribute on <html>
record("P2-lang", "<html lang=...>",
       bool(re.search(r'<html[^>]*\slang=', src)), tower_ref="T8-F1")

# 4 tabs with role="tablist" and role="tab"
tablist = re.findall(r'role="tablist"', src)
tabs = re.findall(r'role="tab"', src)
tabpanels = re.findall(r'role="tabpanel"', src)
record("P2-tablist", f"Has role=tablist ({len(tablist)} found)",
       len(tablist) >= 1, tower_ref="T8-F3")
record("P2-tabs", f"Has 4 role=tab elements ({len(tabs)} found)",
       len(tabs) == 4,
       f"expected 4, found {len(tabs)}" if len(tabs) != 4 else "",
       "T8-F3")
record("P2-tabpanels", f"Has 4 role=tabpanel elements ({len(tabpanels)} found)",
       len(tabpanels) == 4,
       f"expected 4, found {len(tabpanels)}" if len(tabpanels) != 4 else "",
       "T8-F3")

# aria-selected on tabs
aria_selected = re.findall(r'aria-selected="(true|false)"', src)
record("P2-aria-selected", f"Tabs have aria-selected ({len(aria_selected)} attrs)",
       len(aria_selected) >= 4,
       f"only {len(aria_selected)}" if len(aria_selected) < 4 else "",
       "T8-F3")

# aria-controls linking tabs to panels
aria_controls = re.findall(r'aria-controls="(\w+)"', src)
record("P2-aria-controls", f"Tabs have aria-controls ({len(aria_controls)} found)",
       len(aria_controls) >= 4, tower_ref="T8-F3")

# Dialogs have role="dialog" and aria-modal
dialogs = re.findall(r'role="dialog"', src)
aria_modal = re.findall(r'aria-modal="true"', src)
record("P2-dialogs", f"Dialogs have role=dialog ({len(dialogs)} found)",
       len(dialogs) >= 2,
       f"expected >=2 (sign-off + run drawer), found {len(dialogs)}" if len(dialogs) < 2 else "",
       "T8-F5")
record("P2-aria-modal", f"Dialogs have aria-modal ({len(aria_modal)} found)",
       len(aria_modal) >= 2, tower_ref="T8-F5")

# sr-only class used for screen reader content
sr_only = re.findall(r'class="sr-only"', src)
record("P2-sr-only", f"Has sr-only elements ({len(sr_only)} found)",
       len(sr_only) >= 3,
       f"only {len(sr_only)}" if len(sr_only) < 3 else "",
       "T8-F1")

# aria-label on key sections
aria_labels = re.findall(r'aria-label="([^"]+)"', src)
record("P2-aria-labels", f"Has aria-label attributes ({len(aria_labels)} found)",
       len(aria_labels) >= 5,
       f"only {len(aria_labels)}" if len(aria_labels) < 5 else "",
       "T8-F1")

# Form inputs have labels
inputs = re.findall(r'<input\b[^>]*>', src)
selects = re.findall(r'<select\b[^>]*>', src)
visible_inputs = [i for i in inputs if 'type="hidden"' not in i and 'type="submit"' not in i]
for_labels = re.findall(r'<label[^>]*for="([^"]+)"', src)
record("P2-input-labels", f"Inputs/selects have associated labels ({len(for_labels)} labels)",
       len(for_labels) >= len(visible_inputs) + len(selects) - 1,
       f"{len(for_labels)} labels for {len(visible_inputs)} inputs + {len(selects)} selects",
       "T8-F5")

# aria-live regions for dynamic content
aria_live = re.findall(r'aria-live="(polite|assertive)"', src)
record("P2-aria-live", f"Has aria-live regions ({len(aria_live)} found)",
       len(aria_live) >= 2,
       f"only {len(aria_live)}" if len(aria_live) < 2 else "",
       "T8-F7")

# progressbar role
progressbar = re.findall(r'role="progressbar"', src)
record("P2-progressbar", f"Has role=progressbar ({len(progressbar)} found)",
       len(progressbar) >= 1, tower_ref="T8-F3")

PROBE 2: Accessibility (WCAG)
  PASS: <html lang=...>
  PASS: Has role=tablist (1 found)
  PASS: Has 4 role=tab elements (4 found)
  PASS: Has 4 role=tabpanel elements (4 found)
  PASS: Tabs have aria-selected (4 attrs)
  PASS: Tabs have aria-controls (4 found)
  PASS: Dialogs have role=dialog (2 found)
  PASS: Dialogs have aria-modal (2 found)
  PASS: Has sr-only elements (5 found)
  PASS: Has aria-label attributes (13 found)
  PASS: Inputs/selects have associated labels (3 labels)
  PASS: Has aria-live regions (2 found)
  PASS: Has role=progressbar (1 found)


In [4]:
# PROBE 3: Tab Structure — 4 tabs (Upcoming, Approvals, History, eSign) with correct content
# Tower ref: T1 F1 (Jony Ive — purposeful tabs), T23 F31 (Sitemap)
print("=" * 60)
print("PROBE 3: Tab Structure & Content Sections")
print("=" * 60)

src = SCHEDULE_HTML

# Tab 1: Upcoming — has app schedules, keep-alive, calendar
record("P3-tab-upcoming", "Tab: Upcoming exists",
       'data-view="upcoming"' in src, tower_ref="T23-F31")
record("P3-upcoming-apps", "Upcoming: has app schedules section",
       'id="upcomingApps"' in src, tower_ref="T23-F31")
record("P3-upcoming-keepalive", "Upcoming: has keep-alive section",
       'id="upcomingKeepAlive"' in src, tower_ref="T23-F31")
record("P3-upcoming-calendar", "Upcoming: has calendar grid",
       'id="calGrid"' in src, tower_ref="T23-F31")

# Tab 2: Approvals — has kanban board
record("P3-tab-approvals", "Tab: Approvals exists",
       'data-view="approvals"' in src, tower_ref="T23-F31")
record("P3-kanban-waiting", "Approvals: has kanban waiting column",
       'id="kanbanWaiting"' in src, tower_ref="T23-F31")
record("P3-kanban-done", "Approvals: has kanban done column",
       'id="kanbanDone"' in src, tower_ref="T23-F31")

# Tab 3: History — has activity table with correct columns
record("P3-tab-history", "Tab: History exists",
       'data-view="history"' in src, tower_ref="T23-F31")
record("P3-activity-table", "History: has activity table",
       'id="activityTable"' in src, tower_ref="T23-F31")

# Check table headers
th_tags = re.findall(r'<th[^>]*>([^<]+)</th>', src)
expected_cols = {"App", "Status", "Time", "Duration", "Cost", "Evidence", "Flag"}
found_cols = set(t.strip() for t in th_tags)
record("P3-table-cols", f"History: has all 7 columns ({len(found_cols & expected_cols)}/7)",
       expected_cols.issubset(found_cols),
       f"missing: {expected_cols - found_cols}" if not expected_cols.issubset(found_cols) else "",
       "T23-F31")

# Table caption for accessibility
record("P3-table-caption", "History: table has <caption>",
       '<caption' in src, tower_ref="T8-F1")
record("P3-table-scope", "History: <th> elements have scope=col",
       'scope="col"' in src, tower_ref="T8-F1")

# Tab 4: eSign — Part 11 evidence
record("P3-tab-esign", "Tab: eSign exists",
       'data-view="esign"' in src, tower_ref="T23-F31")
record("P3-esign-stats", "eSign: has stats section",
       'id="esignStats"' in src, tower_ref="T23-F31")
record("P3-esign-part11", "eSign: has Part 11 evidence section",
       'id="part11Status"' in src or 'Part 11' in src, tower_ref="T23-F31")

# Savings dashboard
record("P3-savings-dashboard", "Has savings dashboard",
       'id="savingsDashboard"' in src, tower_ref="T23-F31")
record("P3-roi-panel", "Has ROI panel",
       'id="roiPanel"' in src, tower_ref="T23-F31")

# Sign-off queue
record("P3-signoff-alert", "Has sign-off alert",
       'id="signoffAlert"' in src, tower_ref="T23-F31")
record("P3-signoff-sheet", "Has sign-off bottom sheet",
       'id="signoffSheet"' in src, tower_ref="T23-F31")

# Run detail drawer
record("P3-run-drawer", "Has run detail drawer",
       'id="runDrawer"' in src, tower_ref="T23-F31")

# YinYang rail
record("P3-yy-rail", "Has YinYang rail",
       'id="yyRail"' in src, tower_ref="T23-F31")

PROBE 3: Tab Structure & Content Sections
  PASS: Tab: Upcoming exists
  PASS: Upcoming: has app schedules section
  PASS: Upcoming: has keep-alive section
  PASS: Upcoming: has calendar grid
  PASS: Tab: Approvals exists
  PASS: Approvals: has kanban waiting column
  PASS: Approvals: has kanban done column
  PASS: Tab: History exists
  PASS: History: has activity table
  PASS: History: has all 7 columns (7/7)
  PASS: History: table has <caption>
  PASS: History: <th> elements have scope=col
  PASS: Tab: eSign exists
  PASS: eSign: has stats section
  PASS: eSign: has Part 11 evidence section
  PASS: Has savings dashboard
  PASS: Has ROI panel
  PASS: Has sign-off alert
  PASS: Has sign-off bottom sheet
  PASS: Has run detail drawer
  PASS: Has YinYang rail


In [5]:
# PROBE 4: i18n — data-i18n attributes on schedule elements + locale coverage
# Tower ref: T9 (World), T8 (Elders — locale accessibility)
print("=" * 60)
print("PROBE 4: i18n / Translation Coverage")
print("=" * 60)

src = SCHEDULE_HTML

# Count data-i18n attributes
i18n_attrs = re.findall(r'data-i18n="([^"]+)"', src)
i18n_placeholders = re.findall(r'data-i18n-placeholder="([^"]+)"', src)
total_i18n = len(i18n_attrs) + len(i18n_placeholders)

record("P4-i18n-count", f"Schedule page: has data-i18n attrs ({total_i18n})",
       total_i18n >= 5,
       f"only {total_i18n} i18n attrs" if total_i18n < 5 else f"{total_i18n} attrs",
       "T9-F48")

# Key translatable strings that SHOULD be i18n
KEY_STRINGS = {
    "schedule_tab_upcoming": "Upcoming tab",
    "schedule_tab_approvals": "Approvals tab",
    "schedule_tab_history": "History tab",
    "schedule_loading": "Loading text",
    "schedule_status_success": "Success status",
}

for key, desc in KEY_STRINGS.items():
    found = f'data-i18n="{key}"' in src
    record(f"P4-i18n-{key}", f"Has data-i18n for {desc} ({key})",
           found, tower_ref="T9-F48")

# Check locale files have schedule keys
en_path = LOCALES_DIR / "en.json"
if en_path.exists():
    en_data = json.loads(en_path.read_text())

    def flatten_keys(d, prefix=""):
        keys = set()
        for k, v in d.items():
            full = f"{prefix}.{k}" if prefix else k
            if isinstance(v, dict):
                keys.update(flatten_keys(v, full))
            else:
                keys.add(full)
        return keys

    en_keys = flatten_keys(en_data)
    schedule_keys = [k for k in en_keys if "schedule" in k.lower()]
    record("P4-en-schedule-keys", f"English locale has schedule keys ({len(schedule_keys)})",
           len(schedule_keys) >= 3,
           f"only {len(schedule_keys)}: {schedule_keys}" if len(schedule_keys) < 3 else f"{len(schedule_keys)} keys",
           "T9-F48")

    # Check a sample of locales for those same keys
    SAMPLE_LOCALES = ["de", "ja", "ar", "zh", "fr", "es", "ko", "hi"]
    for loc_name in SAMPLE_LOCALES:
        loc_path = LOCALES_DIR / f"{loc_name}.json"
        if loc_path.exists():
            loc_data = json.loads(loc_path.read_text())
            loc_keys = flatten_keys(loc_data)
            has_schedule = any("schedule" in k.lower() for k in loc_keys)
            record(f"P4-locale-{loc_name}", f"{loc_name}: has schedule translation keys",
                   has_schedule, tower_ref="T9-F49")

PROBE 4: i18n / Translation Coverage
  PASS: Schedule page: has data-i18n attrs (5)
  PASS: Has data-i18n for Upcoming tab (schedule_tab_upcoming)
  PASS: Has data-i18n for Approvals tab (schedule_tab_approvals)
  PASS: Has data-i18n for History tab (schedule_tab_history)
  PASS: Has data-i18n for Loading text (schedule_loading)
  PASS: Has data-i18n for Success status (schedule_status_success)
  PASS: English locale has schedule keys (10)
  PASS: de: has schedule translation keys
  PASS: ja: has schedule translation keys
  PASS: ar: has schedule translation keys
  PASS: zh: has schedule translation keys
  PASS: fr: has schedule translation keys
  PASS: es: has schedule translation keys
  PASS: ko: has schedule translation keys
  PASS: hi: has schedule translation keys


In [6]:
# PROBE 5: Security — eval/document.write, escapeHtml in JS modules
# Tower ref: T6 (Hackers), T23 F29 (Security Headers)
# NOTE: Inline styles are acceptable for layout hints; real risks are eval/document.write
print("=" * 60)
print("PROBE 5: Security")
print("=" * 60)

src = SCHEDULE_HTML

# No inline <script> blocks
inline_scripts = re.findall(r'<script(?![^>]*\bsrc=)[^>]*>(.+?)</script>', src, re.DOTALL)
real_inline = [s for s in inline_scripts
               if 'application/ld+json' not in s and len(s.strip()) > 0]
record("P5-no-inline-script", f"No inline <script> blocks ({len(real_inline)} found)",
       len(real_inline) == 0,
       f"{len(real_inline)} inline scripts" if real_inline else "",
       "T6-F1")

# No eval or document.write in page
record("P5-no-eval-html", "No eval() in schedule.html",
       "eval(" not in src, tower_ref="T6-F1")
record("P5-no-docwrite-html", "No document.write() in schedule.html",
       "document.write" not in src, tower_ref="T6-F1")

# Schedule JS modules use escapeHtml
for name, path in SCHEDULE_JS_FILES.items():
    js_src = path.read_text(encoding="utf-8")
    has_innerhtml = ".innerHTML" in js_src
    has_escape = "escapeHtml" in js_src or "textContent" in js_src
    if has_innerhtml:
        record(f"P5-escape-{name}", f"{name}: uses escapeHtml with innerHTML",
               has_escape,
               f"has innerHTML but no escapeHtml/textContent" if not has_escape else "",
               "T6-F1")

# noscript fallback
record("P5-noscript", "Has <noscript> fallback",
       '<noscript>' in src, tower_ref="T8-F1")

# No dangerous patterns in JS
all_js = ""
for path in SCHEDULE_JS_FILES.values():
    all_js += path.read_text(encoding="utf-8") + "\n"

record("P5-no-eval", "Schedule JS: no eval()",
       "eval(" not in all_js, tower_ref="T6-F1")
record("P5-no-document-write", "Schedule JS: no document.write()",
       "document.write" not in all_js, tower_ref="T6-F1")

PROBE 5: Security
  PASS: No inline <script> blocks (0 found)
  PASS: No eval() in schedule.html
  PASS: No document.write() in schedule.html
  PASS: schedule-approvals: uses escapeHtml with innerHTML
  PASS: schedule-calendar: uses escapeHtml with innerHTML
  PASS: schedule-evidence: uses escapeHtml with innerHTML
  PASS: Has <noscript> fallback
  PASS: Schedule JS: no eval()
  PASS: Schedule JS: no document.write()


In [7]:
# EVIDENCE SUMMARY — Convergence check
print("=" * 60)
print("EVIDENCE SUMMARY")
print("=" * 60)

total = len(results)
passed = sum(1 for r in results if r["status"] == "PASS")
failed = sum(1 for r in results if r["status"] == "FAIL")
score = (passed / total * 100) if total else 0
converged = score >= MIN_SCORE

print(f"\nTotal probes:   {total}")
print(f"Passed:         {passed}")
print(f"Failed:         {failed}")
print(f"Score:          {score:.1f}%")
print(f"MIN_SCORE:      {MIN_SCORE}%")
print(f"Converged:      {'YES' if converged else 'NO'}")

if failed > 0:
    print(f"\n{'='*60}")
    print("FAILURES:")
    print(f"{'='*60}")
    for r in results:
        if r["status"] == "FAIL":
            print(f"  FAIL: {r['name']}" + (f" -- {r['detail']}" if r['detail'] else ""))

evidence = json.dumps(results, sort_keys=True)
evidence_hash = hashlib.sha256(evidence.encode()).hexdigest()[:16]
print(f"\nEvidence hash:  {evidence_hash}")
print(f"Timestamp:      {datetime.now().isoformat()}")
print(f"Notebook:       {NOTEBOOK_PATH}")

summary = {
    "northstar": NORTHSTAR,
    "notebook": NOTEBOOK_PATH,
    "total": total,
    "passed": passed,
    "failed": failed,
    "score": round(score, 1),
    "converged": converged,
    "evidence_hash": evidence_hash,
    "findings": [r for r in results if r["status"] == "FAIL"]
}

EVIDENCE SUMMARY

Total probes:   76
Passed:         76
Failed:         0
Score:          100.0%
MIN_SCORE:      70%
Converged:      YES

Evidence hash:  579f5d15355733e2
Timestamp:      2026-03-06T10:24:30.716791
Notebook:       notebooks/qa/01-schedule-page-qa.ipynb
